# Structured Streaming: CSV no volume → Delta, com retomada
Teste finito no AIDP usando trigger availableNow. São duas chegadas de dados sintéticos e três inicializações da mesma consulta/checkpoint.
1. Primeiro arquivo: dois eventos entram na tabela Delta.
2. Novo arquivo: mais dois eventos são processados ao retomar do checkpoint.
3. Sem novos arquivos: retomar mantém quatro eventos sem duplicação.
Não usa Kafka nem mantém tarefa contínua ativa. Não constitui teste de latência ou recuperação de queda abrupta.
Entradas e checkpoint ficam no volume externo existente; a tabela de saída é Delta gerenciada em um schema exclusivo no catálogo default.

In [ ]:
import json, re
from datetime import datetime
from decimal import Decimal
from uuid import uuid4
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, DecimalType, TimestampType
if 'spark' not in globals() or 'oidlUtils' not in globals():
    raise RuntimeError('Execute no runtime AIDP com compute anexado.')
def parameter(name, default):
    value = oidlUtils.parameters.getParameter(name, default)
    return default if value is None else str(value).strip()
def ident(value):
    if not re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*', value):
        raise ValueError(f'Identificador inválido: {value!r}')
    return f'`{value}`'
catalog = parameter('CATALOG_NAME', 'default')
volume_path = parameter('VOLUME_PATH', '/Volumes/default/default/aidp_os_test_volume').rstrip('/')
if not re.fullmatch(r'/Volumes/[A-Za-z_][A-Za-z0-9_]*/[A-Za-z_][A-Za-z0-9_]*/[A-Za-z_][A-Za-z0-9_]*', volume_path):
    raise ValueError('VOLUME_PATH deve apontar para um volume AIDP.')
run_id = uuid4().hex[:12]
schema = f'streaming_demo_{run_id}'
table = '.'.join(ident(s) for s in (catalog, schema, 'transactions_bronze'))
root = f'{volume_path}/streaming/{run_id}'
input_path = root + '/input'
checkpoint_path = root + '/checkpoint'
spark.sql(f'CREATE SCHEMA {ident(catalog)}.{ident(schema)}')
event_schema = StructType([StructField('event_id', StringType()), StructField('account_id', StringType()), StructField('amount', DecimalType(12,2)), StructField('event_time', TimestampType())])
lot1 = [('E001','ACC001',Decimal('120.00'),datetime(2026,11,24,10,0)),('E002','ACC002',Decimal('80.50'),datetime(2026,11,24,10,1))]
lot2 = [('E003','ACC001',Decimal('-25.00'),datetime(2026,11,24,10,2)),('E004','ACC003',Decimal('40.00'),datetime(2026,11,24,10,3))]
result = {'run_id': run_id, 'table':table.replace('`',''), 'input_path':input_path, 'checkpoint_path':checkpoint_path, 'trigger':'availableNow', 'passes':[]}
print('STREAMING_STAGE=setup ' + json.dumps(result))

In [ ]:
def add_csv_file(rows):
    # coalesce(1) produz um arquivo de dados novo por chegada; append não altera arquivos anteriores.
    spark.createDataFrame(rows,event_schema).coalesce(1).write.mode('append').option('header',True).csv(input_path)
def stream_pass(label):
    source = spark.readStream.schema(event_schema).option('header',True).option('mode','FAILFAST').option('maxFilesPerTrigger',1).csv(input_path)
    enriched = source.withColumn('source_file',F.input_file_name()).withColumn('processed_timestamp',F.current_timestamp())
    query = (enriched.writeStream.format('delta').outputMode('append')
        .option('checkpointLocation',checkpoint_path).queryName(f'csv_delta_{run_id}')
        .trigger(availableNow=True).toTable(table))
    try:
        if not query.awaitTermination(120):
            raise TimeoutError('O streaming não terminou em 120 segundos.')
        if query.exception() is not None:
            raise RuntimeError(str(query.exception()))
        progress = query.recentProgress
        passed = {'label':label,'query_id':str(query.id),'query_run_id':str(query.runId),
            'input_rows':sum(int(p.get('numInputRows',0)) for p in progress),
            'batch_ids':[p['batchId'] for p in progress], 'sink_rows':spark.table(table).count()}
        result['passes'].append(passed)
        print('STREAMING_STAGE=pass_complete ' + json.dumps(passed))
    finally:
        if query.isActive:
            query.stop()
def verify(expected):
    actual = sorted((r.event_id,r.account_id,str(r.amount),r.event_time) for r in spark.table(table).select('event_id','account_id','amount','event_time').collect())
    wanted = sorted((r[0],r[1],str(r[2]),r[3]) for r in expected)
    if actual != wanted:
        raise AssertionError(f'Dados divergentes: {actual!r}')

In [ ]:
add_csv_file(lot1)
stream_pass('first_arrival')
verify(lot1)
if result['passes'][-1]['input_rows'] != 2:
    raise AssertionError('O primeiro arquivo deveria produzir dois eventos.')

In [ ]:
add_csv_file(lot2)
stream_pass('resume_with_new_file')
verify(lot1 + lot2)
if result['passes'][-1]['input_rows'] != 2:
    raise AssertionError('A retomada deveria consumir somente os dois eventos novos.')

In [ ]:
# Terceira inicialização sem chegada de arquivos: sem reprocessamento dos arquivos anteriores.
stream_pass('resume_without_new_files')
verify(lot1 + lot2)
if result['passes'][-1]['input_rows'] != 0:
    raise AssertionError('Sem novos arquivos, nenhum evento deveria ser consumido.')
if len({p['query_id'] for p in result['passes']}) != 1:
    raise AssertionError('Esperada a mesma identidade de consulta persistida no checkpoint.')
if len({p['query_run_id'] for p in result['passes']}) != 3:
    raise AssertionError('Esperadas três inicializações distintas da consulta.')
assert spark.table(table).select('event_id').distinct().count() == 4
result.update(status='SUCCESS',rows_verified=4,checkpoint_resume_verified=True,no_replay_duplicates_verified=True,all_queries_stopped=True)
print('AIDP_STREAMING_TEST_RESULT=' + json.dumps(result,sort_keys=True))
spark.table(table).orderBy('event_id').show(truncate=False)

## Recursos retidos
Os dois arquivos CSV, o checkpoint e a tabela Delta permanecem disponíveis para inspeção. Cada execução deste notebook cria um run_id novo; a retomada é testada entre as três inicializações dentro da execução.
A ausência de duplicação comprovada vale para esses arquivos/checkpoint e o sink Delta; não deduplica automaticamente eventos de negócio repetidos em arquivos novos.